# Day 3 lab: AQE and how operators run

**Goal:** Explain how each aggregate and sort executes across partitions, and what Spark changes at runtime once it sees real data sizes.

**Time:** about 2 h 30 min. Checklist: 1.4, 1.9, 1.14, 1.15, 1.16, 3.11 · 🧭 runtime decisions.

Put [`lab_utils.py`](https://github.com/usmanov-saidakbar/python_deepdive/blob/main/public/spark-lab/lab_utils.py) next to this notebook, run the setup cell, then work through the tasks in order. Write your code in the empty cell under each task and your observations in the notes cell. Keep the Spark UI open (the setup cell prints its address).

In [ ]:
from lab_utils import *

spark = get_spark()

## 📘 Code along

Type the code yourself instead of pasting it, run every cell, and keep the Spark UI open. Swap in `spark = get_spark()` from the starter kit wherever an example builds its own session.

### Task 1. Chapter 3: Function Junction — 35 min

Type the cleaning (rename, `na.drop`, `na.fill`, outliers, `distinct`), string, transform and summarizing sections of the official PySpark guide (`explode`, `groupBy().agg()`, `orderBy`); keep the joins section for Day 5. After `groupBy().agg()` and `orderBy`, run `explain()`: partial and final `HashAggregate` around one Exchange, and `rangepartitioning` for the sort. Notice the cell that adds 20 columns with a `withColumn(lit)` loop: Day 8 explains why real code avoids it.

[Chapter 3: Function Junction](https://spark.apache.org/docs/latest/api/python/user_guide/dataprep.html) · [Notebook source on GitHub](https://github.com/apache/spark/blob/v4.2.0/python/docs/source/user_guide/dataprep.ipynb)

In [ ]:
# Task 1: your code here


*Task 1 notes: what you predicted, what the plan and the Spark UI showed.*

## ✍️ Exercises

Write these from scratch in the cells below, using the patterns you just typed. Each one ends with a Check you can verify yourself; open the reference solution only after you have a working answer.

### Task 2. Exercise 1: count three ways — 30 min

On `data/orders`, per status: `count("*")`, the exact `countDistinct("customer_id")`, and `approx_count_distinct("customer_id", rsd)` at 0.05 and 0.01, plus the relative error of each. Keep the exact and the approximate counts in two separate `agg` calls and join them: in one `agg`, Spark computes the sketches inside the distinct count's (status, customer_id) step, about 40,000 groups each holding a 13 KB sketch at rsd 0.01, and the job runs out of memory (try it once and read the plan). With AQE off, count the Exchanges of the exact and the approximate query. 

**Check:** exact is 9,999 for every status, the error is about 4 % at 0.05 and under 1 % at 0.01, and the plans have 2 Exchanges vs 1. Proves 1.9, 1.14.

[approx_count_distinct API](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.approx_count_distinct.html) · [Reference solution](https://github.com/usmanov-saidakbar/python_deepdive/blob/main/public/spark-lab/solutions/day03.py)

In [ ]:
# Task 2: your code here


*Task 2 notes: what you predicted, what the plan and the Spark UI showed.*

### Task 3. Exercise 2: averages and rankings on the chapter's data — 30 min

Rebuild the chapter's `incomes` DataFrame and explode it to one row per income. (1) Compute the overall average income from the rows, and the average of the per-name averages. (2) Top 2 incomes per name with `F.row_number().over(Window.partitionBy("name").orderBy(F.desc("income")))`. (3) The overall top 3 with `orderBy(F.desc("income")).limit(3)`. Compare the two plans with AQE off. 

**Check:** 242.11 vs 244.87, because names have different numbers of incomes; the window plan has an `Exchange hashpartitioning(name)`, a Sort and a Window node; the top 3 is a `TakeOrderedAndProject` with no Exchange. Proves 1.15, 1.16.

[Window API](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.Window.html) · [Reference solution](https://github.com/usmanov-saidakbar/python_deepdive/blob/main/public/spark-lab/solutions/day03.py)

In [ ]:
# Task 3: your code here


*Task 3 notes: what you predicted, what the plan and the Spark UI showed.*

## 🔬 Prove it

Short, focused checks of today's checklist questions in the plan and the Spark UI.

### Task 4. Build the operator table — 30 min

Read `data/orders`, turn AQE off, and for each query save `explain("formatted")` and record the aggregate operator, the number of Exchanges, any Expand, and the sort node: `groupBy("status").agg(F.avg("amount"))`; `.agg(F.countDistinct("customer_id"))`; `.agg(F.countDistinct("customer_id"), F.countDistinct("product_id"))`; `.agg(F.approx_count_distinct("customer_id"))`; `.agg(F.collect_list("amount"))`; `orderBy("amount")`; `sortWithinPartitions("amount")`; `orderBy(F.desc("amount")).limit(10)`. Expect: avg 2 HashAggregate / 1 Exchange; one distinct 4 / 2; two distincts add Expand; approx 2 / 1; collect_list uses ObjectHashAggregate; orderBy has `rangepartitioning` plus `Sort ... true`; sortWithinPartitions has no Exchange; orderBy + limit becomes `TakeOrderedAndProject`. Proves 1.9, 1.14, 1.15, 1.16.

In [ ]:
# Task 4: your code here


*Task 4 notes: what you predicted, what the plan and the Spark UI showed.*

### Task 5. AQE on vs off, and the wrong partition count — 25 min

Query `q = orders.groupBy("customer_id").agg(F.sum("amount"))`. AQE off: run it and count the tasks of the second stage (200); then run it with `spark.sql.shuffle.partitions` at 2 (two big tasks, idle cores) and at 2000 (thousands of tiny tasks). AQE on, partitions still 2000: build `q` again, `q.collect()`, then `q.explain()`: `isFinalPlan=true`, `ShuffleQueryStage`, `AQEShuffleRead coalesced`, and only a few tasks in the Stages tab. Proves 1.4, 3.11.

In [ ]:
# Task 5: your code here


*Task 5 notes: what you predicted, what the plan and the Spark UI showed.*